# 01 - Merge WLASL + MS-ASL Has-Video CSV

Notebook laczy dwa pliki CSV w jednym schemacie uniwersalnym:
- WLASL: `merged_datasets/universal_metadata_has_video.csv`
- MS-ASL: `MS-ASL/msasl_universal_metadata_full_all_raw_has_video.csv`

Wynik zapisywany jest do `merged_datasets/universal_metadata_has_video_wlasl_msasl.csv`.

Dodatkowo notebook normalizuje etykiety przez mapowanie synonimow z `MS-ASL/MSASL_synonym.json` do jednej etykiety kanonicznej.

In [1]:
from pathlib import Path
import json
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)

In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

WLASL_CSV = PROJECT_ROOT / 'merged_datasets' / 'universal_metadata_has_video.csv'
MSASL_CSV = PROJECT_ROOT / 'MS-ASL' / 'msasl_universal_metadata_full_all_raw_has_video.csv'
SYNONYM_JSON = PROJECT_ROOT / 'MS-ASL' / 'MSASL_synonym.json'
OUTPUT_CSV = PROJECT_ROOT / 'merged_datasets' / 'universal_metadata_has_video_wlasl_msasl_merged.csv'

DROP_EXACT_DUPLICATES = False
APPLY_SYNONYM_NORMALIZATION = True

TARGET_COLUMNS = [
    'label',
    'source',
    'video_path',
    'start_frame',
    'end_frame',
    'length_frames',
    'duration_sec',
    'fps',
    'signer_id',
    'has_video',
    'video_width',
    'video_height',
]

print('PROJECT_ROOT:', PROJECT_ROOT)
print('WLASL_CSV exists:', WLASL_CSV.exists(), '|', WLASL_CSV)
print('MSASL_CSV exists:', MSASL_CSV.exists(), '|', MSASL_CSV)
print('SYNONYM_JSON exists:', SYNONYM_JSON.exists(), '|', SYNONYM_JSON)
print('OUTPUT_CSV:', OUTPUT_CSV)
print('DROP_EXACT_DUPLICATES:', DROP_EXACT_DUPLICATES)
print('APPLY_SYNONYM_NORMALIZATION:', APPLY_SYNONYM_NORMALIZATION)

PROJECT_ROOT: D:\college\sem_mag_1\szum
WLASL_CSV exists: True | D:\college\sem_mag_1\szum\merged_datasets\universal_metadata_has_video.csv
MSASL_CSV exists: True | D:\college\sem_mag_1\szum\MS-ASL\msasl_universal_metadata_full_all_raw_has_video.csv
SYNONYM_JSON exists: True | D:\college\sem_mag_1\szum\MS-ASL\MSASL_synonym.json
OUTPUT_CSV: D:\college\sem_mag_1\szum\merged_datasets\universal_metadata_has_video_wlasl_msasl_merged.csv
DROP_EXACT_DUPLICATES: False
APPLY_SYNONYM_NORMALIZATION: True


In [ ]:
def load_and_validate(path: Path, name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f'Missing input file for {name}: {path}')

    df = pd.read_csv(path)
    missing = [col for col in TARGET_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f'{name} CSV is missing required columns: {missing}')

    df = df[TARGET_COLUMNS].copy()

    # Unifikacja booleana dla has_video (np. True/False, 1/0, yes/no).
    as_text = df['has_video'].astype(str).str.strip().str.lower()
    df['has_video'] = as_text.isin(['true', '1', 'yes'])

    return df


def normalize_label(text: str) -> str:
    return str(text).strip().lower()


def build_synonym_map(path: Path) -> dict[str, str]:
    if not path.exists():
        print(f'Synonym file not found, skipping synonym normalization: {path}')
        return {}

    with path.open('r', encoding='utf-8') as f:
        groups = json.load(f)

    syn_map: dict[str, str] = {}
    for group in groups:
        if not isinstance(group, list) or len(group) == 0:
            continue

        canonical = normalize_label(group[0])
        for token in group:
            key = normalize_label(token)
            if key:
                syn_map[key] = canonical

    return syn_map


wlasl_df = load_and_validate(WLASL_CSV, 'WLASL')
msasl_df = load_and_validate(MSASL_CSV, 'MS-ASL')
synonym_map = build_synonym_map(SYNONYM_JSON)

# --- FIX: Create unique signer IDs by adding a source prefix ---
wlasl_df['signer_id'] = wlasl_df['signer_id'].apply(lambda x: f'wlasl_{int(x)}' if pd.notna(x) else None)
msasl_df['signer_id'] = msasl_df['signer_id'].apply(lambda x: f'msasl_{int(x)}' if pd.notna(x) else None)
# ----------------------------------------------------------------

print('Rows WLASL:', len(wlasl_df))
print('Rows MS-ASL:', len(msasl_df))
print('Synonym entries:', len(synonym_map))
display(wlasl_df.head(3))
display(msasl_df.head(3))

Rows WLASL: 11980
Rows MS-ASL: 247
Synonym entries: 588


,label,source,video_path,start_frame,end_frame,length_frames,duration_sec,fps,signer_id,has_video,video_width,video_height
0,book,aslbrick,C:\Users\kacpe\source\repos\szum\kaggle_datase...,1,75.0,75.0,2.502,29.970030,118,True,1280.0,720.0
1,book,signschool,C:\Users\kacpe\source\repos\szum\kaggle_datase...,1,30.0,30.0,1.251,23.976024,31,True,1280.0,720.0
2,book,startasl,C:\Users\kacpe\source\repos\szum\kaggle_datase...,1,68.0,68.0,2.269,29.970000,36,True,736.0,414.0


,label,source,video_path,start_frame,end_frame,length_frames,duration_sec,fps,signer_id,has_video,video_width,video_height
0,match,msasl,D:\college\sem_mag_1\szum\MS-ASL\videos_MS_ASL...,0,83,84.0,2.767,30.0,0,True,640.0,360.0
1,fail,msasl,D:\college\sem_mag_1\szum\MS-ASL\videos_MS_ASL...,0,74,75.0,2.960,25.0,0,True,480.0,360.0
2,book,msasl,D:\college\sem_mag_1\szum\MS-ASL\videos_MS_ASL...,0,66,67.0,2.640,25.0,0,True,480.0,360.0


In [6]:
merged_df = pd.concat([wlasl_df, msasl_df], ignore_index=True)

if APPLY_SYNONYM_NORMALIZATION:
    labels_before = merged_df['label'].astype(str).copy()
    labels_norm = labels_before.str.strip().str.lower()
    merged_df['label'] = labels_norm.map(lambda x: synonym_map.get(x, x))
    changed = (labels_before.str.strip().str.lower() != merged_df['label']).sum()
    print('Labels changed by synonym normalization:', int(changed))

if DROP_EXACT_DUPLICATES:
    before = len(merged_df)
    merged_df = merged_df.drop_duplicates().reset_index(drop=True)
    print('Dropped exact duplicates:', before - len(merged_df))

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
merged_df = merged_df.sort_values(by=['label', 'source'])
merged_df.to_csv(OUTPUT_CSV, index=False)

print('Saved merged CSV:', OUTPUT_CSV)
print('Rows merged:', len(merged_df))
print('Rows by source:')
display(merged_df['source'].value_counts(dropna=False).to_frame('count'))
print('has_video distribution:')
display(merged_df['has_video'].value_counts(dropna=False).to_frame('count'))

Labels changed by synonym normalization: 728
Saved merged CSV: D:\college\sem_mag_1\szum\merged_datasets\universal_metadata_has_video_wlasl_msasl_merged.csv
Rows merged: 12227
Rows by source:


,count
source,
signingsavvy,2668
signschool,1968
aslsearch,1875
asldeafined,1833
spreadthesign,1584
aslsignbank,1050
startasl,623
msasl,247
aslbrick,218


has_video distribution:


,count
has_video,
True,12227


In [7]:
display(merged_df.head(10))
display(merged_df.tail(10))

,label,source,video_path,start_frame,end_frame,length_frames,duration_sec,fps,signer_id,has_video,video_width,video_height
12056,25,msasl,D:\college\sem_mag_1\szum\MS-ASL\videos_MS_ASL...,0,72.0,73.0,2.880,25.000000,0,True,480.0,360.0
12224,50,msasl,D:\college\sem_mag_1\szum\MS-ASL\videos_MS_ASL...,34,89.0,56.0,1.848,29.753000,29,True,540.0,360.0
12225,50,msasl,D:\college\sem_mag_1\szum\MS-ASL\videos_MS_ASL...,89,169.0,81.0,2.689,29.753000,29,True,540.0,360.0
12226,50,msasl,D:\college\sem_mag_1\szum\MS-ASL\videos_MS_ASL...,169,221.0,53.0,1.748,29.753000,29,True,540.0,360.0
8741,a,asldeafined,C:\Users\kacpe\source\repos\szum\kaggle_datase...,1,125.0,125.0,4.167,30.000000,21,True,640.0,480.0
8743,a,aslsearch,C:\Users\kacpe\source\repos\szum\kaggle_datase...,1,84.0,84.0,2.769,30.331450,12,True,720.0,400.0
8740,a,aslsignbank,C:\Users\kacpe\source\repos\szum\kaggle_datase...,1,62.0,62.0,2.586,23.976075,90,True,656.0,370.0
8742,a,signingsavvy,C:\Users\kacpe\source\repos\szum\kaggle_datase...,1,30.0,30.0,1.001,29.970000,11,True,288.0,192.0
8749,a lot,asldeafined,C:\Users\kacpe\source\repos\szum\kaggle_datase...,1,96.0,96.0,3.200,29.996875,15,True,640.0,480.0
8750,a lot,aslsearch,C:\Users\kacpe\source\repos\szum\kaggle_datase...,1,86.0,86.0,2.836,30.322945,12,True,720.0,400.0


,label,source,video_path,start_frame,end_frame,length_frames,duration_sec,fps,signer_id,has_video,video_width,video_height
2320,your,startasl,C:\Users\kacpe\source\repos\szum\kaggle_datase...,1,89.0,89.0,2.970,29.970000,36,True,736.0,414.0
7462,yourself,asldeafined,C:\Users\kacpe\source\repos\szum\kaggle_datase...,1,74.0,74.0,2.466,30.004055,13,True,640.0,480.0
7458,yourself,signingsavvy,C:\Users\kacpe\source\repos\szum\kaggle_datase...,1,31.0,31.0,1.034,29.970000,11,True,288.0,192.0
7460,yourself,signschool,C:\Users\kacpe\source\repos\szum\kaggle_datase...,1,44.0,44.0,1.835,23.976024,32,True,1920.0,1080.0
7461,yourself,signschool,C:\Users\kacpe\source\repos\szum\kaggle_datase...,1,43.0,43.0,1.435,29.970030,4,True,1920.0,1080.0
7459,yourself,spreadthesign,C:\Users\kacpe\source\repos\szum\kaggle_datase...,1,98.0,98.0,1.960,50.000000,26,True,320.0,240.0
6078,zero,aslsearch,C:\Users\kacpe\source\repos\szum\kaggle_datase...,1,57.0,57.0,1.869,30.505723,12,True,720.0,400.0
6079,zero,aslsignbank,C:\Users\kacpe\source\repos\szum\kaggle_datase...,1,57.0,57.0,2.377,23.976108,88,True,656.0,370.0
6081,zero,signingsavvy,C:\Users\kacpe\source\repos\szum\kaggle_datase...,1,30.0,30.0,1.001,29.970000,11,True,288.0,192.0
6080,zero,spreadthesign,C:\Users\kacpe\source\repos\szum\kaggle_datase...,1,51.0,51.0,2.040,25.000000,1,True,320.0,240.0
